In [1]:
import pandas as pd
import re

In [ ]:
df = pd.read_csv('data\\car_features_dataset.csv')



In [4]:
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

Number of duplicate rows: 715


In [5]:
duplicate_rows_df = df[df.duplicated()]
print(duplicate_rows_df.head(5))

    Make     Model  Year             Engine Fuel Type  Engine HP  \
14   BMW  1 Series  2013  premium unleaded (required)      230.0   
18  Audi       100  1992             regular unleaded      172.0   
20  Audi       100  1992             regular unleaded      172.0   
24  Audi       100  1993             regular unleaded      172.0   
25  Audi       100  1993             regular unleaded      172.0   

    Engine Cylinders Transmission Type      Driven_Wheels  Number of Doors  \
14               6.0            MANUAL   rear wheel drive              2.0   
18               6.0            MANUAL  front wheel drive              4.0   
20               6.0            MANUAL  front wheel drive              4.0   
24               6.0            MANUAL  front wheel drive              4.0   
25               6.0            MANUAL  front wheel drive              4.0   

       Market Category Vehicle Size Vehicle Style  highway MPG  city mpg  \
14  Luxury,Performance      Compact         Co

In [6]:
df_cleaned = df.drop_duplicates()
print(f"Cleaned file saved! Remaining rows: {len(df_cleaned)}")

Cleaned file saved! Remaining rows: 11199


In [8]:

def clean_market_category(category):
    """Clean and format market category field"""
    if pd.isna(category) or category in ["N/A", ""]:
        return ""
    # Remove quotes and split by comma
    categories = str(category).replace('"', '').split(',')
    return ', '.join(categories).lower()

def format_vehicle_description(row):
    """Format a single row into natural language description"""
    
    # Clean and format fields
    year = int(row['Year']) if pd.notna(row['Year']) else ""
    make = str(row['Make']).strip()
    model = str(row['Model']).strip()
    
    # Format engine info
    engine_fuel = str(row['Engine Fuel Type']).replace(' (required)', '').replace(' (recommended)', '').lower()
    engine_hp = int(row['Engine HP']) if pd.notna(row['Engine HP']) else "unknown"
    engine_cyl = int(row['Engine Cylinders']) if pd.notna(row['Engine Cylinders']) else "unknown"
    
    # Transmission type
    transmission = str(row['Transmission Type']).lower()
    
    # Drivetrain
    driven_wheels = str(row['Driven_Wheels']).replace(' wheel drive', '').lower()
    
    # Door count
    doors = int(row['Number of Doors']) if pd.notna(row['Number of Doors']) else ""
    
    # Market category
    market_cat = clean_market_category(row['Market Category'])
    
    # Vehicle size and style
    vehicle_size = str(row['Vehicle Size']).lower()
    vehicle_style = str(row['Vehicle Style']).lower()
    
    # MPG
    hwy_mpg = int(row['highway MPG']) if pd.notna(row['highway MPG']) else "unknown"
    city_mpg = int(row['city mpg']) if pd.notna(row['city mpg']) else "unknown"
    
    # Popularity
    popularity = int(row['Popularity']) if pd.notna(row['Popularity']) else ""
    
    # MSRP
    msrp = row['MSRP']
    if pd.notna(msrp):
        msrp_str = f"${msrp:,.0f}"
    else:
        msrp_str = "unknown price"
    
    # Build the description
    description = f"The {year} {make} {model} is a {vehicle_size} {vehicle_style}.\n"
    
    # Engine and drivetrain info
    description += f"It features a {engine_cyl}-cylinder {engine_fuel} engine producing {engine_hp} horsepower.\n"
    
    # Transmission and drivetrain
    description += f"The transmission is {transmission} with {driven_wheels} drive.\n"
    
    # Additional features
    if doors:
        description += f"It has {doors} doors.\n"
    
    if market_cat:
        description += f"This model falls under the following categories: {market_cat}.\n"
    
    # Fuel efficiency
    if hwy_mpg != "unknown" or city_mpg != "unknown":
        mpg_str = []
        if city_mpg != "unknown":
            mpg_str.append(f"{city_mpg} city MPG")
        if hwy_mpg != "unknown":
            mpg_str.append(f"{hwy_mpg} highway MPG")
        description += f"Fuel efficiency is rated at {' and '.join(mpg_str)}.\n"
    
    # Popularity and MSRP
    if popularity:
        description += f"It has a popularity score of {popularity}.\n"
    
    description += f"The MSRP for this vehicle is {msrp_str}.\n"
    
    return description

def csv_to_knowledge_base(csv_file_path, output_file_path):
    """Convert CSV data to knowledge base text file"""
    
    # Read the CSV file
    df = pd.read_csv(csv_file_path)

    # Remove duplicates
    df_cleaned = df.drop_duplicates()
    
    # Create output text
    output_lines = []
    
    # Process each row
    for idx, row in df.iterrows():
        description = format_vehicle_description(row)
        output_lines.append(description)
        output_lines.append("")  # Add empty line between entries
    
    # Write to text file
    with open(output_file_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(output_lines))
    
    print(f"Successfully converted {len(df)} records to {output_file_path}")
    print(f"Sample output (first vehicle):\n{'-'*50}")
    print(format_vehicle_description(df.iloc[0]))



In [10]:
# Usage
csv_file_path = "data\\car_features_dataset.csv"  # Replace with your CSV file path
output_file_path = "data\\vehicle_knowledge_base.txt"

csv_to_knowledge_base(csv_file_path, output_file_path)

Successfully converted 11914 records to data\vehicle_knowledge_base.txt
Sample output (first vehicle):
--------------------------------------------------
The 2011 BMW 1 Series M is a compact coupe.
It features a 6-cylinder premium unleaded engine producing 335 horsepower.
The transmission is manual with rear drive.
It has 2 doors.
This model falls under the following categories: factory tuner, luxury, high-performance.
Fuel efficiency is rated at 19 city MPG and 26 highway MPG.
It has a popularity score of 3916.
The MSRP for this vehicle is $46,135.

